In [4]:
import logging
import pandas as pd
from pathlib import Path
logging.basicConfig(level=logging.INFO,format="%(levelname)s | %(message)s")
log=logging.getLogger(__name__)

BASE_DIR=Path().resolve().parent
RAW_DIR= BASE_DIR / "data" / "raw"
PROC_DIR=BASE_DIR / "data" / "processed"
PROC_DIR.mkdir(parents=True,exist_ok=True)

In [5]:
nav=pd.read_csv(RAW_DIR / "02_nav_history.csv")
print(nav.shape)
nav.head()

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
nav['date']=pd.to_datetime(nav['date'])
nav.sort_values(['amfi_code', 'date'], inplace=True)
nav = nav[nav['nav'] > 0]
nav.drop_duplicates(subset=['amfi_code', 'date'], inplace=True)
nav.to_csv(PROC_DIR / "clean_nav.csv", index=False)
print(f"clean_nav.csv saved — {nav.shape}")

clean_nav.csv saved — (46000, 3)


In [9]:
txn=pd.read_csv(RAW_DIR/"08_investor_transactions.csv")
print(txn.shape)
txn.head()

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [10]:
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])
txn['transaction_type'] = txn['transaction_type'].str.strip().str.title()
txn = txn[txn['transaction_type'].isin(['Sip', 'Lumpsum', 'Redemption'])]
txn = txn[txn['amount_inr'] > 0]
txn.drop_duplicates(inplace=True)
txn.to_csv(PROC_DIR / "clean_transactions.csv", index=False)
print(f"clean_transactions.csv saved — {txn.shape}")

clean_transactions.csv saved — (32778, 13)


In [11]:
perf = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
print(perf.shape)
perf.head()

(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [13]:
for col in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

print("Negative Sharpe:", perf[perf['sharpe_ratio'] < 0].shape[0])
print("Expense out of range:", perf[~perf['expense_ratio_pct'].between(0.1, 2.5)].shape[0])

perf.drop_duplicates(inplace=True)
perf.to_csv(PROC_DIR / "clean_performance.csv", index=False)
print(f"clean_performance.csv saved — {perf.shape}")

Negative Sharpe: 0
Expense out of range: 0
clean_performance.csv saved — (40, 19)


In [16]:
import sqlite3
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

BASE_DIR = Path().resolve().parent
PROC_DIR = BASE_DIR / "data" / "processed"
DB_PATH  = BASE_DIR / "data" / "db" / "bluestock_mf.db"

engine = create_engine(f"sqlite:///{DB_PATH}")
print("Connected to DB")

Connected to DB


In [18]:
fund = pd.read_csv(PROC_DIR / "clean_performance.csv")[
    ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'expense_ratio_pct', 'risk_grade']
]
fund.to_sql("dim_fund", engine, if_exists="replace", index=False)
print(f"dim_fund loaded — {len(fund)} rows")

dim_fund loaded — 40 rows


In [19]:
nav = pd.read_csv(PROC_DIR / "clean_nav.csv")
nav.rename(columns={"date": "nav_date"}, inplace=True)
nav.to_sql("fact_nav", engine, if_exists="replace", index=False)
print(f"fact_nav loaded — {len(nav)} rows")

fact_nav loaded — 46000 rows


In [20]:
txn = pd.read_csv(PROC_DIR / "clean_transactions.csv")
txn.to_sql("fact_transactions", engine, if_exists="replace", index=False)
print(f"fact_transactions loaded — {len(txn)} rows")

fact_transactions loaded — 32778 rows


In [21]:
perf = pd.read_csv(PROC_DIR / "clean_performance.csv")[
    ['amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
     'sharpe_ratio', 'beta', 'alpha', 'aum_crore']
]
perf.to_sql("fact_performance", engine, if_exists="replace", index=False)
print(f"fact_performance loaded — {len(perf)} rows")

fact_performance loaded — 40 rows


In [22]:
conn = sqlite3.connect(DB_PATH)
for table in ["dim_fund", "fact_nav", "fact_transactions", "fact_performance"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")
conn.close()

dim_fund: 40 rows
fact_nav: 46000 rows
fact_transactions: 32778 rows
fact_performance: 40 rows


In [23]:
conn = sqlite3.connect(DB_PATH)

queries = open(BASE_DIR / "sql" / "queries.sql").read().split(";")
for i, q in enumerate(queries):
    q = q.strip()
    if q:
        print(f"\n--- Query {i+1} ---")
        print(pd.read_sql(q, conn))

conn.close()


--- Query 1 ---
                                         scheme_name         fund_house  \
0  Mirae Asset Emerging Bluechip Fund - Regular -...     Mirae Asset MF   
1      Kotak Emerging Equity Fund - Regular - Growth  Kotak Mahindra MF   
2     Nippon India Small Cap Fund - Regular - Growth    Nippon India MF   
3         DSP Top 100 Equity Fund - Regular - Growth    DSP Mutual Fund   
4                UTI Mid Cap Fund - Regular - Growth    UTI Mutual Fund   

   aum_crore  
0      49046  
1      47469  
2      43630  
3      41828  
4      41728  

--- Query 2 ---
      amfi_code    month  avg_nav
0        100016  2022-01   512.54
1        100016  2022-02   513.93
2        100016  2022-03   522.58
3        100016  2022-04   525.63
4        100016  2022-05   504.31
...         ...      ...      ...
2115     149324  2026-01   253.51
2116     149324  2026-02   249.43
2117     149324  2026-03   263.05
2118     149324  2026-04   305.94
2119     149324  2026-05   303.72

[2120 rows x 3 c